# SOLUSDT order hedging check

Assumptions for hedging success:
- Match open and hedge orders by `strategy_id`.
- Required hedge qty = open `filled_qty` (not the original order qty).
- Hedged ok when sum of hedge `filled_qty` >= required qty (within a small tolerance).
- Rows with open `filled_qty == 0` are excluded from the unhedged list.


In [1]:
import pandas as pd

engine_used = "pyarrow"
try:
    df = pd.read_parquet("SOLUSDT_order.parquet", engine="pyarrow")
except Exception:
    engine_used = "fastparquet"
    df = pd.read_parquet("SOLUSDT_order.parquet", engine="fastparquet")

print(f"engine={engine_used}, rows={len(df)}, cols={df.shape[1]}")

df.head()


engine=fastparquet, rows=3346, cols=22


,symbol,order_id,client_order_id,side,order_type,price,quantity,status,trading_venue,strategy_id,...,order_side,create_ts,opening_bid0,opening_ask0,hedging_bid0,hedging_ask0,price_offset,update_ts,local_ts,filled_qty
0,SOL-USDT-SWAP,3147551042519949312,6143485400171151361,SELL,LIMIT,125.72,1.59,CANCELED,OkexFutures,1430391660,...,open,1.766307e+15,125.68,125.69,125.65,125.67,0.0003,1766306766053000000,1766306730900000000,0.0
1,SOL-USDT-SWAP,3147551042352177159,6143483600579854337,SELL,LIMIT,125.71,1.59,CANCELED,OkexFutures,1430391241,...,open,1.766307e+15,125.68,125.69,125.65,125.67,0.0001,1766306766048000000,1766306730895000000,0.0
2,SOL-USDT-SWAP,3147550225435975682,6038895073838497793,SELL,LIMIT,126.05,1.58,CANCELED,OkexFutures,1406039827,...,open,1.766307e+15,125.78,125.79,125.75,125.76,0.0020,1766306766053000000,1766306706549000000,0.0
3,SOL-USDT-SWAP,3147550225435975681,6038893111038443521,SELL,LIMIT,125.94,1.58,CANCELED,OkexFutures,1406039370,...,open,1.766307e+15,125.78,125.79,125.75,125.76,0.0012,1766306766053000000,1766306706549000000,0.0
4,SOL-USDT-SWAP,3147550225435975680,6038891062339043329,SELL,LIMIT,125.88,1.58,CANCELED,OkexFutures,1406038893,...,open,1.766307e+15,125.78,125.79,125.75,125.76,0.0007,1766306766053000000,1766306706549000000,0.0


In [2]:
open_df = df[df["order_side"] == "open"].copy()
hedge_df = df[df["order_side"] == "hedge"].copy()

open_df = open_df.rename(
    columns={
        "order_id": "open_order_id",
        "client_order_id": "open_client_order_id",
        "status": "open_status",
        "side": "open_side",
        "quantity": "open_qty",
        "filled_qty": "open_filled_qty",
        "create_ts": "open_create_ts",
        "update_ts": "open_update_ts",
        "local_ts": "open_local_ts",
    }
)

open_cols = [
    "symbol",
    "open_order_id",
    "open_client_order_id",
    "open_side",
    "order_type",
    "price",
    "open_qty",
    "open_status",
    "trading_venue",
    "strategy_id",
    "opening_venue",
    "hedging_venue",
    "open_create_ts",
    "open_update_ts",
    "open_local_ts",
    "open_filled_qty",
]

open_df = open_df[open_cols].set_index("strategy_id")

hedge_agg = hedge_df.groupby("strategy_id").agg(
    hedge_order_count=("order_id", "size"),
    hedge_filled_count=("status", lambda s: (s == "FILLED").sum()),
    hedge_canceled_count=("status", lambda s: (s == "CANCELED").sum()),
    hedge_filled_qty_sum=("filled_qty", "sum"),
)

summary = open_df.join(hedge_agg, how="left")

for col in ["hedge_order_count", "hedge_filled_count", "hedge_canceled_count"]:
    summary[col] = summary[col].fillna(0).astype(int)
summary["hedge_filled_qty_sum"] = summary["hedge_filled_qty_sum"].fillna(0.0)

summary.head()


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,hedging_venue,open_create_ts,open_update_ts,open_local_ts,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum
strategy_id,,,,,,,,,,,,,,,,,,,
1430391660,SOL-USDT-SWAP,3147551042519949312,6143485400171151361,SELL,LIMIT,125.72,1.59,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766307e+15,1766306766053000000,1766306730900000000,0.0,0,0,0,0.0
1430391241,SOL-USDT-SWAP,3147551042352177159,6143483600579854337,SELL,LIMIT,125.71,1.59,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766307e+15,1766306766048000000,1766306730895000000,0.0,0,0,0,0.0
1406039827,SOL-USDT-SWAP,3147550225435975682,6038895073838497793,SELL,LIMIT,126.05,1.58,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766307e+15,1766306766053000000,1766306706549000000,0.0,0,0,0,0.0
1406039370,SOL-USDT-SWAP,3147550225435975681,6038893111038443521,SELL,LIMIT,125.94,1.58,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766307e+15,1766306766053000000,1766306706549000000,0.0,0,0,0,0.0
1406038893,SOL-USDT-SWAP,3147550225435975680,6038891062339043329,SELL,LIMIT,125.88,1.58,CANCELED,OkexFutures,OkexFutures,BinanceFutures,1.766307e+15,1766306766053000000,1766306706549000000,0.0,0,0,0,0.0


In [3]:
summary["need_hedge_qty"] = summary["open_filled_qty"]

tol = 1e-9
summary["hedged_ok"] = summary["hedge_filled_qty_sum"] + tol >= summary["need_hedge_qty"]

need_hedge = summary[summary["need_hedge_qty"] > 0]
unhedged = need_hedge[~need_hedge["hedged_ok"]].sort_values("open_create_ts")

print(f"open rows: {len(open_df)}")
print(f"need hedge rows: {len(need_hedge)}")
print(f"unhedged rows: {len(unhedged)}")

unhedged


open rows: 1916
need hedge rows: 334
unhedged rows: 8


,symbol,open_order_id,open_client_order_id,open_side,order_type,price,open_qty,open_status,trading_venue,opening_venue,...,open_create_ts,open_update_ts,open_local_ts,open_filled_qty,hedge_order_count,hedge_filled_count,hedge_canceled_count,hedge_filled_qty_sum,need_hedge_qty,hedged_ok
strategy_id,,,,,,,,,,,,,,,,,,,,,
1870436056,SOL-USDT-SWAP,3147133462378438656,8033461689779224577,SELL,LIMIT,125.27,0.79,FILLED,OkexFutures,OkexFutures,...,1.766294e+15,1766294295375000000,1766294286042000000,0.79,0,0,0,0.00,0.79,False
1870435656,SOL-USDT-SWAP,3147133462210666496,8033459971792306177,SELL,LIMIT,125.25,0.79,FILLED,OkexFutures,OkexFutures,...,1.766294e+15,1766294286601000000,1766294286037000000,0.79,0,0,0,0.00,0.79,False
1879419837,SOL-USDT-SWAP,3147133763697238016,8072046735368650753,SELL,LIMIT,125.29,0.79,FILLED,OkexFutures,OkexFutures,...,1.766294e+15,1766294313966000000,1766294295022000000,0.79,0,0,0,0.00,0.79,False
1678035792,SOL-USDT-SWAP,3147343179155496960,7207108848157458433,SELL,LIMIT,125.35,1.59,FILLED,OkexFutures,OkexFutures,...,1.766301e+15,1766300562650000000,1766300536089000000,1.59,4,1,3,0.82,1.59,False
1813025122,SOL-USDT-SWAP,3147491823980634112,7786883605816410113,SELL,LIMIT,124.91,1.60,FILLED,OkexFutures,OkexFutures,...,1.766305e+15,1766304969076000000,1766304966050000000,1.60,3,0,3,0.00,1.60,False
1813026643,SOL-USDT-SWAP,3147491823913525249,7786890138461667329,SELL,LIMIT,125.05,1.59,FILLED,OkexFutures,OkexFutures,...,1.766305e+15,1766305034337000000,1766304966048000000,1.59,0,0,0,0.00,1.59,False
1813026169,SOL-USDT-SWAP,3147491823913525248,7786888102647169025,SELL,LIMIT,124.99,1.60,FILLED,OkexFutures,OkexFutures,...,1.766305e+15,1766304989639000000,1766304966048000000,1.60,0,0,0,0.00,1.60,False
1818068997,SOL-USDT-SWAP,3147491993061416960,7808546883986522113,SELL,LIMIT,124.94,1.60,FILLED,OkexFutures,OkexFutures,...,1.766305e+15,1766304977932000000,1766304971089000000,1.60,0,0,0,0.00,1.60,False


In [4]:
def show_strategy(strategy_id: int):
    return df[df["strategy_id"] == strategy_id].sort_values(
        ["order_side", "create_ts", "order_id"]
    )

# Example: show_strategy(1870436056)
